# 00 · Environment and setup

**Project:** *Traffic Flow Prediction and Signal Control via Graph Attention Networks and Flow Models*

**Research question.** Does incorporating spatial relationships between connected traffic sensors improve
multi-step traffic-speed forecasting compared with models that treat every sensor independently?

**Data.** METR-LA: 5-minute speed readings from 207 loop-detector sensors on Los Angeles highways
(1 Mar - 27 Jun 2012), plus the road-network distance graph published with
[DCRNN](https://github.com/liyaguang/DCRNN).

**Models compared** (identical chronological split, windows, targets, and evaluation rules):

| Model | Uses the road graph? | Notebook |
|---|---|---|
| Persistence, rolling mean, seasonal average | no | 04 |
| XGBoost (global, direct multi-horizon, per-sensor features only) | no | 05 |
| ST-GAT (GRU encoder + graph attention + multi-horizon head) and its no-edge ablation | yes / no | 06 |
| DCRNN (optional) | yes | 06b |

**This notebook** only inspects and prepares the environment; it does not modify any installed package.
Signal control is **simulated only** (notebook 09, optional); nothing here connects to real traffic infrastructure
or live feeds.

## Notebook map and execution order

| # | Notebook | Produces |
|---|---|---|
| 00 | `00_environment_and_setup` | environment report |
| 01 | `01_data_exploration` | data understanding, figures |
| 02 | `02_data_preprocessing` | `data/processed/{traffic_clean.npz, splits.json, scaler.json}` |
| 03 | `03_graph_construction` | `data/processed/{adjacency.npy, sensor_order.csv, graph_meta.json}` |
| 04 | `04_baseline_models` | baseline predictions + metrics |
| 05 | `05_xgboost_forecasting` | XGBoost model + predictions |
| 06 | `06_graph_attention_network` | trained ST-GAT (+ ablation) + predictions |
| 06b | `06b_optional_dcrnn` | optional DCRNN comparison |
| 07 | `07_model_evaluation` | all comparison tables / figures |
| 08 | `08_congestion_visualization` | geographic maps |
| 09 | `09_optional_sumo_simulation` | optional simulated signal control |

Every notebook is **restart-safe**: it reloads what it needs from `data/processed/` and `artifacts/` instead of
relying on variables from another notebook. Set the environment variable `TFC_RUN_MODE=smoke` *before starting
Jupyter* to run everything on tiny subsets (outputs get a `_smoke` suffix and must not be reported as results).

## 1 · Methodology

1. Report the Python / OS / hardware environment.
2. Show (but do not execute) the installation commands.
3. Import the project package, fix random seeds, and print the central configuration.
4. Detect CUDA / Apple-MPS / CPU and run a tiny tensor test on the chosen device.
5. Create the directory tree, configure the dataset paths, and check that every required raw file exists.
6. Report package versions and finish with a PASS / FAIL setup summary.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)

# Locate the project root (the folder that contains src/config.py) wherever Jupyter was started.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))

from src import config
config.ensure_dirs()
print(f"project root : {ROOT}")
print(f"run mode     : {config.RUN_MODE}" + ("   <-- SMOKE TEST: outputs are NOT final results" if config.IS_SMOKE else ""))

project root : /Users/ishanmaheshwari1/Desktop/AML-DLProject/traffic-flow-control
run mode     : full


## 2 · Python and platform

In [2]:
import platform, sys, os

print("python        :", sys.version.split()[0], f"({sys.executable})")
print("platform      :", platform.platform())
print("machine       :", platform.machine(), "| processor:", platform.processor() or "n/a")
print("logical cores :", os.cpu_count())
in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
print("virtual env   :", sys.prefix if in_venv else "not using a virtual environment")
apple_silicon = platform.system() == "Darwin" and platform.machine() == "arm64"
print("Apple Silicon :", apple_silicon)

python        : 3.13.6 (/Users/ishanmaheshwari1/Desktop/AML-DLProject/traffic-flow-control/.venv/bin/python3.13)
platform      : macOS-27.0.1-arm64-arm-64bit-Mach-O
machine       : arm64 | processor: arm
logical cores : 8
virtual env   : /Users/ishanmaheshwari1/Desktop/AML-DLProject/traffic-flow-control/.venv
Apple Silicon : True


## 3 · Dependency installation (optional, **not** executed automatically)

Only run this if packages are missing. It installs into the *active* environment, so create a virtual environment first:

```bash
cd traffic-flow-control
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

macOS note: the XGBoost wheel needs the OpenMP runtime. If `import xgboost` fails with a `libomp.dylib` error run
`brew install libomp`.

The cell below is commented out on purpose so that re-running this notebook never changes a working environment.

In [3]:
# Uncomment ONLY if packages are missing (installs into the active kernel's environment):
# %pip install -r "{ROOT}/requirements.txt"

## 4 · Imports and reproducibility configuration

All hyper-parameters, horizons, split fractions and paths live in `src/config.py`. The seed is applied to Python,
NumPy and PyTorch. (GPU kernels, especially on Apple MPS and CUDA, are not guaranteed bit-wise deterministic.)

In [4]:
import random, numpy as np, pandas as pd

try:
    import torch
except ImportError as exc:
    raise SystemExit("PyTorch is required: see the installation cell above") from exc

random.seed(config.SEED); np.random.seed(config.SEED); torch.manual_seed(config.SEED)

settings = {
    "seed": config.SEED,
    "input window": f"{config.INPUT_LEN} steps = {config.INPUT_LEN * config.STEP_MINUTES} min",
    "horizons (steps)": config.HORIZONS,
    "horizons (minutes)": tuple(config.HORIZON_MINUTES[h] for h in config.HORIZONS),
    "split (train/val/test)": f"{config.TRAIN_FRAC:.0%} / {config.VAL_FRAC:.0%} / {1 - config.TRAIN_FRAC - config.VAL_FRAC:.0%} (day aligned)",
    "missing-value sentinel": config.MISSING_SENTINEL,
    "MAPE minimum true speed (mph)": config.MAPE_MIN_SPEED,
    "peak windows (local hours, weekdays)": config.PEAK_WINDOWS,
    "assumed time zone": config.DATA_TIMEZONE,
}
for k, v in settings.items():
    print(f"{k:38s}: {v}")

seed                                  : 42
input window                          : 12 steps = 60 min
horizons (steps)                      : (1, 3, 6, 12)
horizons (minutes)                    : (5, 15, 30, 60)
split (train/val/test)                : 70% / 10% / 20% (day aligned)
missing-value sentinel                : 0.0
MAPE minimum true speed (mph)         : 1.0
peak windows (local hours, weekdays)  : ((7, 10), (15, 19))
assumed time zone                     : America/Los_Angeles


## 5 · CPU / GPU detection

The graph model picks **CUDA** if available, otherwise **Apple MPS**, otherwise the **CPU**. Nothing assumes a GPU exists.

In [5]:
from src.gat_model import select_device

device = select_device()
print("torch version      :", torch.__version__)
print("CUDA available     :", torch.cuda.is_available())
print("MPS built/available:", torch.backends.mps.is_built(), "/", torch.backends.mps.is_available())
print("CPU threads (torch):", torch.get_num_threads())
print("selected device    :", device)

# Tiny functional test: a matmul + GRU forward/backward on the chosen device.
a = torch.randn(256, 256, device=device)
gru = torch.nn.GRU(8, 8, batch_first=True).to(device)
out, h = gru(torch.randn(4, 12, 8, device=device))
(out.sum() + (a @ a).sum() * 0).backward()
print("device smoke test  : OK")

torch version      : 2.14.1
CUDA available     : False
MPS built/available: True / True
CPU threads (torch): 4
selected device    : mps


device smoke test  : OK


## 6 · Project directories and dataset paths

In [6]:
config.ensure_dirs()
for name in ["DATA_RAW", "SENSOR_GRAPH_DIR", "DATA_PROCESSED", "MODELS_DIR", "PREDICTIONS_DIR", "FIGURES_DIR", "METRICS_DIR", "REPORTS_DIR"]:
    p = getattr(config, name)
    print(f"{name:18s} {str(p.relative_to(config.PROJECT_ROOT)):28s} exists={p.is_dir()}")

DATA_RAW           data/raw                     exists=True
SENSOR_GRAPH_DIR   data/raw/sensor_graph        exists=True
DATA_PROCESSED     data/processed               exists=True
MODELS_DIR         artifacts/models             exists=True
PREDICTIONS_DIR    artifacts/predictions        exists=True
FIGURES_DIR        artifacts/figures            exists=True
METRICS_DIR        artifacts/metrics            exists=True
REPORTS_DIR        reports                      exists=True


## 7 · Dataset availability

`require_raw_files()` fails with an actionable message listing every missing file. The files are **not** committed
to Git (see `data/README.md` for sources and checksums).

In [7]:
import hashlib
from src import data_loader as dl

rows = []
for path, description in config.REQUIRED_RAW_FILES.items():
    exists = path.is_file()
    rows.append({
        "file": str(path.relative_to(config.PROJECT_ROOT)),
        "present": exists,
        "size_MB": round(path.stat().st_size / 1e6, 3) if exists else None,
        "sha256[:12]": hashlib.sha256(path.read_bytes()).hexdigest()[:12] if exists else None,
    })
availability = pd.DataFrame(rows)
display(availability)

missing = dl.missing_raw_files()
if missing:
    print("MISSING FILES:")
    for p, d in missing.items():
        print(f"  - {p.relative_to(config.PROJECT_ROOT)}  ({d})")
    print("See data/README.md for download instructions; the remaining notebooks cannot run without them.")
else:
    print("All required raw files are present.")

,file,present,size_MB,sha256[:12]
0,data/raw/metr-la.h5,True,57.038,64784b76d6fb
1,data/raw/sensor_graph/graph_sensor_ids.txt,True,0.001,3ba026caa2e6
2,data/raw/sensor_graph/graph_sensor_locations.csv,True,0.006,eb8ea96e0735
3,data/raw/sensor_graph/distances_la_2012.csv,True,6.393,a576a2a3e28d


All required raw files are present.


## 8 · Package versions

In [8]:
from importlib import metadata

packages = ["numpy", "pandas", "scipy", "scikit-learn", "h5py", "torch", "torch_geometric", "xgboost",
            "networkx", "matplotlib", "seaborn", "plotly", "folium", "nbformat", "nbclient", "pytest"]
versions = {}
for pkg in packages:
    try:
        versions[pkg] = metadata.version(pkg)
    except metadata.PackageNotFoundError:
        versions[pkg] = "NOT INSTALLED"
display(pd.Series(versions, name="version").to_frame())

# xgboost can be installed but fail to load without the OpenMP runtime (macOS).
try:
    import xgboost
    xgb_ok, xgb_msg = True, f"xgboost {xgboost.__version__} loads"
except Exception as exc:  # noqa: BLE001 - we want the reason, whatever it is
    xgb_ok, xgb_msg = False, f"xgboost import failed: {str(exc).strip().splitlines()[0][:120]} (macOS: `brew install libomp`)"
print(xgb_msg)

import shutil
print("SUMO (optional, notebook 09):", shutil.which("sumo") or "not found on PATH")

,version
numpy,2.5.4
pandas,3.0.6
scipy,1.18.1
scikit-learn,1.9.1
h5py,3.16.0
torch,2.14.1
torch_geometric,2.8.1
xgboost,3.4.1
networkx,3.7
matplotlib,3.11.2


xgboost 3.4.1 loads
SUMO (optional, notebook 09): not found on PATH


## 9 · Setup verification summary

Every *required* check must pass before continuing. SUMO is optional and reported separately.

In [9]:
import importlib.util

checks = {
    "Python >= 3.10": sys.version_info >= (3, 10),
    "required packages installed": all(v != "NOT INSTALLED" for k, v in versions.items() if k not in {"torch_geometric", "folium", "seaborn"}),
    "xgboost loads": xgb_ok,
    "torch device usable": True,  # the smoke test above would have raised otherwise
    "all raw data files present": not missing,
    "project directories exist": all(getattr(config, n).is_dir() for n in ["DATA_PROCESSED", "MODELS_DIR", "PREDICTIONS_DIR", "FIGURES_DIR", "METRICS_DIR", "REPORTS_DIR"]),
}
summary = pd.DataFrame({"check": list(checks), "status": ["PASS" if v else "FAIL" for v in checks.values()]})
display(summary)
print("Optional: PyTorch Geometric installed ->", importlib.util.find_spec("torch_geometric") is not None,
      "(only needed by the layer-equivalence unit test)")
print("Optional: SUMO on PATH               ->", shutil.which("sumo") is not None)

assert all(checks.values()), "Setup incomplete - fix the FAIL rows above before running the next notebooks."
print("\nSETUP VERIFIED - continue with 01_data_exploration.ipynb")

,check,status
0,Python >= 3.10,PASS
1,required packages installed,PASS
2,xgboost loads,PASS
3,torch device usable,PASS
4,all raw data files present,PASS
5,project directories exist,PASS


Optional: PyTorch Geometric installed -> True (only needed by the layer-equivalence unit test)
Optional: SUMO on PATH               -> False

SETUP VERIFIED - continue with 01_data_exploration.ipynb


### Interpretation

The environment is ready when every row above reads `PASS`. If a data file is missing, notebooks 01-08 raise a
`FileNotFoundError` with the same instructions rather than silently skipping a step. Next: **01 · data exploration**.